In [0]:
import usaddress

# usaddress tags that together form the street line, in output order.
STREET_PARTS = [
    "AddressNumberPrefix", 
    "AddressNumber", 
    "AddressNumberSuffix",
    "StreetNamePreDirectional", 
    "StreetNamePreModifier", 
    "StreetNamePreType",
    "StreetName", 
    "StreetNamePostType", 
    "StreetNamePostDirectional",
    "StreetNamePostModifier",
]

# Output contract. Config builds BRONZE_STRUCT from this list, so the parser
# owns its own schema rather than config dictating it.
PARSED_COLUMNS = ["Address_1", "Address_2", "City", "State", "Zip_Code", "Parsing_Error"]

# Assign None to all PARSED_COLUMNS
_EMPTY_PARSE = {c: None for c in PARSED_COLUMNS}

def parse_address(address):
    """Parse a US address string into component columns.

    Args:
        address (str | None): Raw address line.

    Returns:
        dict: Always contains all six PARSED_COLUMNS keys.
            Address_1 (str | None): Street line assembled from STREET_PARTS.
            Address_2 (str | None): Suite type and number; None without a
                suite number.
            City (str | None): Parsed place name.
            State (str | None): Parsed state.
            Zip_Code (str | None): Parsed ZIP.
            Parsing_Error (str | None): Failure reason, None on success.

    Note:
        A None Parsing_Error does not imply a correct parse. Every token is
        assigned some label, so an ambiguous address succeeds with wrong
        values. Validate City and State downstream.
    """
    if not address:
        return {**_EMPTY_PARSE, "Parsing_Error": "Empty input"}

    try:
        tagged, _ = usaddress.tag(address)
    except usaddress.RepeatedLabelError:
        return {**_EMPTY_PARSE, "Parsing_Error": "RepeatedLabelError"}
    except Exception as e:
        return {**_EMPTY_PARSE, "Parsing_Error": str(e)}

    street = " ".join(p for p in (tagged.get(k) for k in STREET_PARTS) if p)
    suite_type = tagged.get("OccupancyType")
    suite_number = tagged.get("OccupancyIdentifier")

    return {
        "Address_1": street or None,
        "Address_2": " ".join(p for p in (suite_type, suite_number) if p) if suite_number else None,
        "City": tagged.get("PlaceName"),
        "State": tagged.get("StateName"),
        "Zip_Code": tagged.get("ZipCode"),
        "Parsing_Error": None
    }